<a href="https://colab.research.google.com/github/alianas-dev/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alianas-dev/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/alianas-dev/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found — check the repo cloned correctly"
print("Found it. You're good to go.")

Working dir: /content/flyrank-ml-internship
Found it. You're good to go.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Refresh / Content Opportunity Scoring. I frame it as a ranking task because the goal is not simply to label every page as good or bad. The goal is to rank content pages by how strongly they appear to need review, so a limited review team can start with the highest-priority pages. The ranking can use multiple observable signals such as impressions, sessions, content age, CTR, position, and trend.

The question "which pages should an editor look at first?" maps to ranking / scoring, not classification: a classifier would force a hard yes/no cut at some arbitrary threshold, but a reviewer with limited time per week just needs an ordered queue, and the boundary between "review now" and "review later" is a capacity question, not a property of the page itself.

In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Rows (pages) to rank: {len(df):,}")
print(f"Distinct clients: {df['client_id'].nunique()}")
print("A binary classifier would draw one hard line through all of these pages.")
print("A ranking/scoring approach orders them instead — which fits a review queue better than a single yes/no cut.")

Rows (pages) to rank: 30,000
Distinct clients: 32
A binary classifier would draw one hard line through all of these pages.
A ranking/scoring approach orders them instead — which fits a review queue better than a single yes/no cut.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For this starter exercise, I will use `trend_direction == "down"` as a proxy target for content that may need review. This label comes from an observed trend in the current data window rather than a future outcome. Therefore, it is a proxy for potential decline, not proof that a page will decline or that refreshing it will improve performance. A stronger future version would use features from a prior time window to predict an outcome in a later window.

Important gotcha (from the data dictionary): `trend_direction` is itself computed from `trend_pct`, which is computed from `impressions_last_30d` and `impressions_prev_30d`. So the proxy label and its ingredients can never be used as model *features* later — only as the thing being predicted.

In [3]:
label = (df["trend_direction"] == "down").astype(int)

print("trend_direction value counts:")
print(df["trend_direction"].value_counts())
print()
print(f"Proxy label rate (is_declining_label == 1): {label.mean():.1%} of the {len(df):,} pages")
print("This label is observed in the current 90-day window, not a future outcome — a proxy, not a guarantee.")

trend_direction value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Proxy label rate (is_declining_label == 1): 54.2% of the 30,000 pages
This label is observed in the current 90-day window, not a future outcome — a proxy, not a guarantee.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I will use Precision@50 as the main success metric. The goal is to rank pages for a limited review queue, so the quality of the top 50 recommendations matters more than overall accuracy. A higher Precision@50 means that a larger proportion of the highest-ranked pages match the chosen proxy target. For example, a Precision@50 of 0.60 would mean that 30 of the top 50 ranked pages match the proxy label.

I am choosing Precision@50 over accuracy because accuracy would reward correctly labeling the ~46% of pages that are *not* declining, which the reviewer never looks at — it does not tell the reviewer whether their actual weekly queue is any good. Precision@K is defensible because it is measured directly against what the reviewer will act on: the top of the queue.

In [4]:
# Illustrate the metric mechanics with one simple, non-model ranking: worst CTR first.
# This is only to show HOW Precision@50 is computed, not a claim that this rule is good
# (Section 5 tests whether simple rules like this actually work).
ranked_by_ctr = df.assign(is_declining_label=label).sort_values("ctr", ascending=True)
top_50 = ranked_by_ctr.head(50)

precision_at_50 = top_50["is_declining_label"].mean()
matches = int(top_50["is_declining_label"].sum())

print(f"Example — ranking by lowest CTR first:")
print(f"Precision@50 = {precision_at_50:.2f}  ->  {matches} of the top 50 ranked pages match the proxy label")
print(f"(Base rate if we ranked randomly would be about {label.mean():.2f})")

Example — ranking by lowest CTR first:
Precision@50 = 0.50  ->  25 of the top 50 ranked pages match the proxy label
(Base rate if we ranked randomly would be about 0.54)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page (one `content_id`) belonging to one client (`client_id`), with its search and analytics metrics aggregated over a trailing 90-day window ending at export time. There is no repetition: `content_id` is unique per row, so a page is not tracked over multiple time points in this starter slice — it is one snapshot per page.

In [5]:
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique content_id: {df['content_id'].nunique():,}  (equals row count -> one row per page, confirmed)")
print(f"Unique client_id: {df['client_id'].nunique()}  (32 pseudonymized clients sharing this dataframe)")
print()

cols_to_show = [
    "content_id", "client_id", "content_type", "content_age_days",
    "impressions_90d", "sessions_90d", "ctr", "avg_position",
    "trend_direction", "trend_pct",
]
df[cols_to_show].head(5)

Shape: 30,000 rows x 44 columns
Unique content_id: 30,000  (equals row count -> one row per page, confirmed)
Unique client_id: 32  (32 pseudonymized clients sharing this dataframe)



,content_id,client_id,content_type,content_age_days,impressions_90d,sessions_90d,ctr,avg_position,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,187,3803,17,0.76,10.6,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,15320,9,0.05,20.3,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,12581,11,0.09,36.5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,463,11751,78,0.49,6.2,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,19140,145,0.13,44.0,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

No single observable signal correlates strongly with the proxy decline label — the strongest single correlate, `content_age_days`, is still weak. And when I hand-write "obvious" if-statement rules using multiple signals together (older content, worse average position), the rule does not clearly beat doing nothing — in this slice it actually lands *below* the base rate. That is the real argument for ML over a fixed rule: the signals only carry information in combination, the combination is not a simple AND/OR of thresholds, and the "right" combination is not something I can guess by eye. A model can learn the weighted interaction across many signals (impressions, sessions, CTR, position, age, freshness, content type) that no single hand-picked if-statement captures — which matches the roughly 3x lift the project's baseline pipeline sees when the model (Precision@50 ≈ 0.74) replaces the hand-written rule (Precision@50 ≈ 0.24).

There is also structural messiness beneath the numbers: missingness is systematic rather than random (e.g. `feedly article` pages have no keyword/search-volume data at all, while `keyword article` pages mostly do), so a blind rule risks encoding content type by accident rather than genuinely capturing decline risk.

In [6]:
candidate_signals = [
    "engagement_rate", "ctr", "avg_position", "word_count", "content_age_days",
    "days_since_last_update", "scroll_rate", "ai_traffic_pct", "impressions_90d", "sessions_90d",
]
corr_with_label = df[candidate_signals].corrwith(pd.Series(label, index=df.index)).sort_values(key=abs, ascending=False)
print("Correlation of individual signals with the proxy label (none of these is strong alone):")
print(corr_with_label.round(3))
print()

# A hand-written two-signal if-statement rule: "old content AND weak average position"
age_thresh = df["content_age_days"].median()
pos_thresh = df.loc[df["avg_position"] > 0, "avg_position"].median()
rule_flag = (df["content_age_days"] > age_thresh) & (df["avg_position"] > pos_thresh)

rule_precision = label[rule_flag].mean()
rule_recall = label[rule_flag].sum() / label.sum()

print(f"If-statement rule (age > {age_thresh:.0f} days AND avg_position > {pos_thresh:.1f}):")
print(f"  flags {rule_flag.sum():,} pages -> precision {rule_precision:.3f}, recall {rule_recall:.3f}")
print(f"  base rate (flag nothing / guess randomly): {label.mean():.3f}")
print("  -> the hand rule does NOT clearly beat the base rate, even combining two 'obvious' signals.")
print()

missing_by_type = df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean())
print("Share of rows missing search_volume, by content_type (systematic, not random):")
print(missing_by_type.round(3))

Correlation of individual signals with the proxy label (none of these is strong alone):
content_age_days         -0.164
word_count                0.090
days_since_last_update    0.081
ctr                      -0.062
avg_position             -0.029
sessions_90d             -0.023
impressions_90d          -0.018
engagement_rate          -0.013
scroll_rate              -0.003
ai_traffic_pct            0.002
dtype: float64

If-statement rule (age > 236 days AND avg_position > 11.4):
  flags 7,225 pages -> precision 0.460, recall 0.204
  base rate (flag nothing / guess randomly): 0.542
  -> the hand rule does NOT clearly beat the base rate, even combining two 'obvious' signals.

Share of rows missing search_volume, by content_type (systematic, not random):
content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014
Name: search_volume, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.